# Triton 内存与数据搬运 - 课后练习

本 Notebook 包含三个练习，难度递进，帮助你巩固内存管理知识。

**学习目标**：
- 掌握 2D 地址计算和 stride 处理
- 理解内存连续性对性能的影响
- 学会优化数据复用，减少重复加载
- 使用 cache hints 优化访存性能
- 处理复杂的边界情况

In [ ]:
import torch
import triton
import triton.language as tl
import time

# 检查 GPU 可用性
assert torch.cuda.is_available(), "需要 CUDA 支持的 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Triton version: {triton.__version__}")

---

## 练习 1: 优化 2D 卷积

**目标**：实现一个高效的 2D 卷积（3×3 box filter），要求：
1. 使用数据复用技术，减少重复加载
2. 正确处理边界和 padding
3. 支持 stride

**提示**：
1. 参考教程中的 `conv1d_optimized` 方法
2. 加载 `(BLOCK_SIZE_H + 2) × (BLOCK_SIZE_W + 2)` 的块
3. 使用切片操作获取邻居数据
4. 注意边界处理

In [ ]:
@triton.jit
def conv2d_box_filter_kernel(
    input_ptr, output_ptr,
    H, W,
    stride_h, stride_w,
    BLOCK_SIZE_H: tl.constexpr,
    BLOCK_SIZE_W: tl.constexpr,
):
    """
    TODO: 实现优化的 2D 卷积
    Y[i][j] = sum(X[i-1:i+2][j-1:j+2])
    
    步骤：
    1. 计算 Program ID
    2. 加载 (BLOCK_SIZE_H + 2) × (BLOCK_SIZE_W + 2) 的块
    3. 创建边界 mask
    4. 使用切片获取 3×3 邻居
    5. 求和
    6. 存储结果
    """
    # ==================== 在下方编写代码 ====================
    
    
    
    # ========================================================
    pass

def conv2d_box_filter(input_tensor):
    """
    Host 端包装函数
    
    Args:
        input_tensor: (H, W) 的 tensor
    
    Returns:
        (H, W) 的输出 tensor
    """
    H, W = input_tensor.shape
    output = torch.empty_like(input_tensor)
    
    # Grid 配置
    grid = lambda meta: (
        triton.cdiv(H, meta['BLOCK_SIZE_H']),
        triton.cdiv(W, meta['BLOCK_SIZE_W']),
    )
    
    # 启动 kernel
    conv2d_box_filter_kernel[grid](
        input_tensor, output,
        H, W,
        input_tensor.stride(0), input_tensor.stride(1),
        BLOCK_SIZE_H=64,
        BLOCK_SIZE_W=64,
    )
    
    return output

In [ ]:
# 测试 2D 卷积
def test_conv2d():
    H, W = 512, 512
    input_tensor = torch.randn(H, W, device='cuda', dtype=torch.float32)
    
    # Triton 实现
    output_triton = conv2d_box_filter(input_tensor)
    
    # PyTorch 参考实现
    output_torch = torch.nn.functional.avg_pool2d(
        input_tensor.unsqueeze(0), 
        kernel_size=3, 
        stride=1, 
        padding=1
    ).squeeze(0) * 9  # avg_pool2d 会除以 9，所以乘回去
    
    # 验证
    if torch.allclose(output_triton, output_torch, atol=1e-4):
        print("✓ 2D 卷积测试通过！")
    else:
        print("✗ 2D 卷积测试失败！")
        print(f"最大误差: {torch.max(torch.abs(output_triton - output_torch)).item():.2e}")
        print(f"\n前 5x5 元素对比:")
        print(f"Triton:\n{output_triton[:5, :5]}")
        print(f"Torch:\n{output_torch[:5, :5]}")

test_conv2d()

In [ ]:
# 性能对比
def benchmark_conv2d():
    sizes = [(256, 256), (512, 512), (1024, 1024)]
    
    print(f"{'Size':>15} | {'Triton (ms)':>12} | {'Torch (ms)':>12} | {'Speedup':>10}")
    print("-" * 60)
    
    for H, W in sizes:
        input_tensor = torch.randn(H, W, device='cuda', dtype=torch.float32)
        
        # Triton 实现
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(100):
            output = conv2d_box_filter(input_tensor)
        torch.cuda.synchronize()
        t_triton = (time.time() - t0) * 1000
        
        # PyTorch 实现
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(100):
            output_torch = torch.nn.functional.avg_pool2d(
                input_tensor.unsqueeze(0), kernel_size=3, stride=1, padding=1
            ).squeeze(0) * 9
        torch.cuda.synchronize()
        t_torch = (time.time() - t0) * 1000
        
        speedup = t_torch / t_triton
        print(f"{H}x{W:>10} | {t_triton:>12.2f} | {t_torch:>12.2f} | {speedup:>10.2f}x")

benchmark_conv2d()

**思考题**：
1. 如果卷积核大小变为 5×5，代码需要如何修改？
2. 如何使用 cache_modifier 进一步优化性能？
3. 这种方法和 CUDA 的 Shared Memory 有什么异同？

**提示**（点击展开）：

<details>
<summary>提示 1：如何计算加载块的坐标</summary>

加载块需要包含 padding，所以要从 -1 开始：
```python
# 加载块的坐标
h_load = pid_h * BLOCK_SIZE_H + tl.arange(0, BLOCK_SIZE_H + 2) - 1
w_load = pid_w * BLOCK_SIZE_W + tl.arange(0, BLOCK_SIZE_W + 2) - 1
```
</details>

<details>
<summary>提示 2：如何创建 mask</summary>

需要检查加载的坐标是否在有效范围内：
```python
mask = (h_load[:, None] >= 0) & (h_load[:, None] < H) & \
       (w_load[None, :] >= 0) & (w_load[None, :] < W)
```
</details>

<details>
<summary>提示 3：如何切片</summary>

使用切片获取 3×3 窗口：
```python
# 3×3 邻居求和
result = (
    input_block[:-2, :-2] + input_block[:-2, 1:-1] + input_block[:-2, 2:] +
    input_block[1:-1, :-2] + input_block[1:-1, 1:-1] + input_block[1:-1, 2:] +
    input_block[2:, :-2] + input_block[2:, 1:-1] + input_block[2:, 2:]
)
```
</details>

## 练习 2: 优化矩阵乘法内存访问

**目标**：实现一个高效的矩阵乘法 kernel，运用 cache hints 优化访存

**提示**：
1. 参考教程中的 `matmul_with_cache_hints`
2. 使用 `cache_modifier` 参数优化加载
3. 注意 stride 的正确传递
4. 处理非 BLOCK_SIZE 整数倍的情况

In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    """
    TODO: 实现带 cache hints 的矩阵乘法
    
    步骤：
    1. 获取 Program ID
    2. 初始化累加器
    3. 分块循环加载 A 和 B（使用 cache_modifier）
    4. 计算 dot product
    5. 存储结果
    """
    # ==================== 在下方编写代码 ====================
    
    
    
    # ========================================================
    pass

def matmul(a, b):
    """
    Host 端包装函数
    
    Args:
        a: (M, K) tensor
        b: (K, N) tensor
    
    Returns:
        (M, N) tensor
    """
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, "矩阵维度不匹配"
    
    c = torch.empty(M, N, device=a.device, dtype=a.dtype)
    
    # Grid 配置
    grid = lambda meta: (
        triton.cdiv(M, meta['BLOCK_SIZE_M']) * triton.cdiv(N, meta['BLOCK_SIZE_N']),
    )
    
    # 启动 kernel
    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_SIZE_M=64,
        BLOCK_SIZE_N=64,
        BLOCK_SIZE_K=32,
    )
    
    return c

In [ ]:
# 测试矩阵乘法
def test_matmul():
    M, N, K = 512, 512, 512
    
    a = torch.randn(M, K, device='cuda', dtype=torch.float32)
    b = torch.randn(K, N, device='cuda', dtype=torch.float32)
    
    # Triton 实现
    c_triton = matmul(a, b)
    
    # PyTorch 参考实现
    c_torch = torch.matmul(a, b)
    
    # 验证
    if torch.allclose(c_triton, c_torch, atol=1e-3):
        print("✓ 矩阵乘法测试通过！")
    else:
        print("✗ 矩阵乘法测试失败！")
        print(f"最大误差: {torch.max(torch.abs(c_triton - c_torch)).item():.2e}")

test_matmul()

In [ ]:
# 性能对比：有/无 cache hints
def benchmark_matmul_cache_hints():
    sizes = [(512, 512, 512), (1024, 1024, 1024), (2048, 2048, 2048)]
    
    print(f"{'Size':>15} | {'No hint (ms)':>13} | {'With hint (ms)':>14} | {'Speedup':>10}")
    print("-" * 65)
    
    # 实现：无 cache hints 的版本
    @triton.jit
    def matmul_no_hint(
        a_ptr, b_ptr, c_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        BLOCK_SIZE_M: tl.constexpr,
        BLOCK_SIZE_N: tl.constexpr,
        BLOCK_SIZE_K: tl.constexpr,
    ):
        pid = tl.program_id(0)
        pid_m = pid // (triton.cdiv(N, BLOCK_SIZE_N))
        pid_n = pid % (triton.cdiv(N, BLOCK_SIZE_N))
        
        acc = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
        
        for k in range(0, K, BLOCK_SIZE_K):
            a = tl.load(a_ptr + ..., mask=..., other=0.0)  # 无 cache hint
            b = tl.load(b_ptr + ..., mask=..., other=0.0)  # 无 cache hint
            acc += tl.dot(a, b)
        
        tl.store(c_ptr + ..., acc, mask=...)
    
    for M, N, K in sizes:
        a = torch.randn(M, K, device='cuda', dtype=torch.float32)
        b = torch.randn(K, N, device='cuda', dtype=torch.float32)
        
        # 测试无 cache hint
        c1 = torch.empty(M, N, device='cuda')
        grid = lambda meta: (triton.cdiv(M, meta['BLOCK_SIZE_M']) * triton.cdiv(N, meta['BLOCK_SIZE_N']),)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(10):
            matmul_no_hint[grid](a, b, c1, M, N, K, a.stride(0), a.stride(1), b.stride(0), b.stride(1), c1.stride(0), c1.stride(1), 64, 64, 32)
        torch.cuda.synchronize()
        t_no_hint = (time.time() - t0) * 1000
        
        # 测试有 cache hint
        c2 = matmul(a, b)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(10):
            _ = matmul(a, b)
        torch.cuda.synchronize()
        t_with_hint = (time.time() - t0) * 1000
        
        speedup = t_no_hint / t_with_hint
        print(f"{M}x{K}x{N:>6} | {t_no_hint:>13.2f} | {t_with_hint:>14.2f} | {speedup:>10.2f}x")

benchmark_matmul_cache_hints()

**思考题**：
1. cache_modifier 对不同大小的矩阵影响是否相同？为什么？
2. 什么情况下 `.ca` 和 `.cg` 的效果差异最大？
3. 如何处理非方阵的情况？

**提示**（点击展开）：

<details>
<summary>提示 1：Program ID 计算</summary>

```python
pid = tl.program_id(0)
pid_m = pid // (triton.cdiv(N, BLOCK_SIZE_N))
pid_n = pid % (triton.cdiv(N, BLOCK_SIZE_N))
```
</details>

<details>
<summary>提示 2：使用 cache hints</summary>

```python
a = tl.load(a_ptrs, mask=mask, other=0.0, cache_modifier=".ca")
b = tl.load(b_ptrs, mask=mask, other=0.0, cache_modifier=".cg")
```
</details>

<details>
<summary>提示 3：指针计算</summary>

```python
rm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
rn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
rk = k * BLOCK_SIZE_K + tl.arange(0, BLOCK_SIZE_K)

a_ptrs = a_ptr + (rm[:, None] * stride_am + rk[None, :] * stride_ak)
b_ptrs = b_ptr + (rk[:, None] * stride_bk + rn[None, :] * stride_bn)
```
</details>

## 练习 3: 带 Cache Hints 的 Softmax

**目标**：实现一个高效的 Softmax kernel，要求：
1. 两阶段 reduction（先求 max，再求 sum）
2. 使用 cache_modifier 优化访存
3. 支持任意长度的输入序列

**背景**：
Softmax 计算步骤：
1. 找到最大值：`max_x = max(x)`
2. 计算 exp：`exp_x = exp(x - max_x)`
3. 求和：`sum_exp = sum(exp_x)`
4. 归一化：`softmax = exp_x / sum_exp`

**难点**：
- 步骤 1 需要跨元素 reduction
- 步骤 3 又需要一次 reduction
- 如何高效地共享中间结果？

In [ ]:
@triton.jit
def softmax_kernel(
    input_ptr, output_ptr,
    n_rows, n_cols,
    stride_row, stride_col,
    BLOCK_SIZE: tl.constexpr,
):
    """
    TODO: 实现带 cache hints 的 Softmax
    
    步骤：
    1. 获取当前行
    2. 第一次 reduction：找 max
    3. 计算 exp(x - max)
    4. 第二次 reduction：求 exp 和
    5. 归一化
    6. 存储结果
    """
    # ==================== 在下方编写代码 ====================
    
    
    
    # ========================================================
    pass

def softmax(x):
    """
    Host 端包装函数
    
    Args:
        x: (n_rows, n_cols) tensor
    
    Returns:
        (n_rows, n_cols) softmax 结果
    """
    n_rows, n_cols = x.shape
    output = torch.empty_like(x)
    
    # Grid 配置
    grid = lambda meta: (n_rows,)
    
    # 启动 kernel
    softmax_kernel[grid](
        x, output,
        n_rows, n_cols,
        x.stride(0), x.stride(1),
        BLOCK_SIZE=1024,
    )
    
    return output

In [ ]:
# 测试 Softmax
def test_softmax():
    n_rows, n_cols = 32, 2048
    x = torch.randn(n_rows, n_cols, device='cuda', dtype=torch.float32)
    
    # Triton 实现
    y_triton = softmax(x)
    
    # PyTorch 参考实现
    y_torch = torch.nn.functional.softmax(x, dim=-1)
    
    # 验证
    if torch.allclose(y_triton, y_torch, atol=1e-4):
        print("✓ Softmax 测试通过！")
    else:
        print("✗ Softmax 测试失败！")
        print(f"最大误差: {torch.max(torch.abs(y_triton - y_torch)).item():.2e}")

test_softmax()

In [ ]:
# 性能对比
def benchmark_softmax():
    sizes = [(32, 512), (32, 1024), (32, 2048), (32, 4096)]
    
    print(f"{'Rows':>8} | {'Cols':>8} | {'Triton (ms)':>12} | {'Torch (ms)':>12} | {'Speedup':>10}")
    print("-" * 65)
    
    for n_rows, n_cols in sizes:
        x = torch.randn(n_rows, n_cols, device='cuda', dtype=torch.float32)
        
        # Triton 实现
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(100):
            output = softmax(x)
        torch.cuda.synchronize()
        t_triton = (time.time() - t0) * 1000
        
        # PyTorch 实现
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(100):
            output_torch = torch.nn.functional.softmax(x, dim=-1)
        torch.cuda.synchronize()
        t_torch = (time.time() - t0) * 1000
        
        speedup = t_torch / t_triton
        print(f"{n_rows:>8} | {n_cols:>8} | {t_triton:>12.2f} | {t_torch:>12.2f} | {speedup:>10.2f}x")

# benchmark_softmax()  # 取消注释以运行

**思考题**（高级）：
1. 如何处理数值稳定性？（提示：exp(x - max_x)）
2. 如何优化 reduction 的性能？（提示：两级 reduction）
3. 这种方法能扩展到 2D Softmax 吗？

**提示**（点击展开）：

<details>
<summary>提示 1：Reduction 模式</summary>

```python
# 使用 tl.max 和 tl.sum 进行 reduction
row = tl.program_id(0)
col_offsets = tl.arange(0, BLOCK_SIZE)
mask = col_offsets < n_cols

# 加载当前行
x = tl.load(input_ptr + row * stride_row + col_offsets * stride_col, mask=mask, other=float('-inf'))

# 找最大值
x_max = tl.max(x, axis=0)

# 计算 exp(x - max)
x_exp = tl.exp(x - x_max)

# 求和
x_sum = tl.sum(x_exp, axis=0)

# 归一化
y = x_exp / x_sum
```
</details>

<details>
<summary>提示 2：使用 cache hints</summary>

```python
# 输入数据会被多次访问（找 max、求 sum），使用 .cg
x = tl.load(input_ptr + ..., mask=mask, other=float('-inf'), cache_modifier=".cg")
```
</details>


## 总结

完成这三个练习后，你应该掌握了：
- 2D/多维地址计算和 mask 处理
- 数据复用优化技术
- Cache hints 的实际应用
- Reduction 操作的实现方法

**下一步**：学习 Triton 的 Reduction 与原子操作！

---

## 课后答案

### 练习 1：2D 卷积

```python
@triton.jit
def conv2d_box_filter_kernel(
    input_ptr, output_ptr,
    H, W,
    stride_h, stride_w,
    BLOCK_SIZE_H: tl.constexpr,
    BLOCK_SIZE_W: tl.constexpr,
):
    pid_h = tl.program_id(axis=0)
    pid_w = tl.program_id(axis=1)
    
    # 加载 (BLOCK_SIZE_H + 2) × (BLOCK_SIZE_W + 2) 的块
    h = pid_h * BLOCK_SIZE_H + tl.arange(0, BLOCK_SIZE_H + 2) - 1
    w = pid_w * BLOCK_SIZE_W + tl.arange(0, BLOCK_SIZE_W + 2) - 1
    
    # 创建边界 mask
    mask = (h[:, None] >= 0) & (h[:, None] < H) & \
           (w[None, :] >= 0) & (w[None, :] < W)
    
    # 加载块
    input_ptrs = input_ptr + (h[:, None] * stride_h + w[None, :] * stride_w)
    input_block = tl.load(input_ptrs, mask=mask, other=0.0)
    
    # 3×3 邻居求和
    result = (
        input_block[:-2, :-2] + input_block[:-2, 1:-1] + input_block[:-2, 2:] +
        input_block[1:-1, :-2] + input_block[1:-1, 1:-1] + input_block[1:-1, 2:] +
        input_block[2:, :-2] + input_block[2:, 1:-1] + input_block[2:, 2:]
    )
    
    # 存储结果
    out_h = pid_h * BLOCK_SIZE_H + tl.arange(0, BLOCK_SIZE_H)
    out_w = pid_w * BLOCK_SIZE_W + tl.arange(0, BLOCK_SIZE_W)
    out_mask = (out_h[:, None] < H) & (out_w[None, :] < W)
    
    output_ptrs = output_ptr + (out_h[:, None] * stride_h + out_w[None, :] * stride_w)
    tl.store(output_ptrs, result, mask=out_mask)
```

### 练习 2：矩阵乘法

```python
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    pid = tl.program_id(0)
    pid_m = pid // (triton.cdiv(N, BLOCK_SIZE_N))
    pid_n = pid % (triton.cdiv(N, BLOCK_SIZE_N))
    
    rm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    rn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    
    # 初始化累加器
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    
    # 分块计算
    for k in range(0, K, BLOCK_SIZE_K):
        rk = k * BLOCK_SIZE_K + tl.arange(0, BLOCK_SIZE_K)
        
        # 创建 mask
        mask_m = rm[:, None] < M
        mask_n = rn[None, :] < N
        mask_k = rk[None, :] < K
        mask = mask_m & mask_n & mask_k
        
        # 计算指针
        a_ptrs = a_ptr + (rm[:, None] * stride_am + rk[None, :] * stride_ak)
        b_ptrs = b_ptr + (rk[:, None] * stride_bk + rn[None, :] * stride_bn)
        
        # 使用 cache hints 优化加载
        a = tl.load(a_ptrs, mask=mask, other=0.0, cache_modifier=".ca")
        b = tl.load(b_ptrs, mask=mask, other=0.0, cache_modifier=".cg")
        
        # 累加
        accumulator += tl.dot(a, b)
    
    # 存储结果
    cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    
    out_mask = (cm[:, None] < M) & (cn[None, :] < N)
    c_ptrs = c_ptr + (cm[:, None] * stride_cm + cn[None, :] * stride_cn)
    
    tl.store(c_ptrs, accumulator, mask=out_mask)
```

### 练习 3：Softmax

```python
@triton.jit
def softmax_kernel(
    input_ptr, output_ptr,
    n_rows, n_cols,
    stride_row, stride_col,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(0)
    col_offsets = tl.arange(0, BLOCK_SIZE)
    
    # Mask
    mask = col_offsets < n_cols
    
    # 加载当前行（使用 cache hints）
    x = tl.load(
        input_ptr + row * stride_row + col_offsets * stride_col,
        mask=mask,
        other=float('-inf'),
        cache_modifier=".cg"
    )
    
    # 找最大值（数值稳定性）
    x_max = tl.max(x)
    
    # 计算 exp(x - max)
    x_exp = tl.exp(x - x_max)
    
    # 求和
    x_sum = tl.sum(x_exp)
    
    # 归一化
    y = x_exp / x_sum
    
    # 存储结果
    tl.store(
        output_ptr + row * stride_row + col_offsets * stride_col,
        y,
        mask=mask
    )
```